In [1]:
# ============================================================
#   🐱🐶  CATS VS DOGS – TRANSFER LEARNING CLASSIFIER
#          Mini Project – Day 16
#          MobileNetV2 + Feature Extraction + Fine-Tuning
# ============================================================

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

IMG_SIZE   = 160
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE
CLASS_NAMES = ['Cat', 'Dog']

print("=" * 64)
print("   🐱🐶  CATS vs DOGS – TRANSFER LEARNING CLASSIFIER  🐱🐶")
print("=" * 64)

# ── STEP 1: Load Dataset ─────────────────────────────────────
print("\n📦 STEP 1: Loading Cats vs Dogs Dataset (TFDS)...")

dataset, info = tfds.load(
    "cats_vs_dogs",
    with_info=True,
    as_supervised=True
)

# The full dataset is under 'train' key; we split manually
full_ds     = dataset['train']
total       = info.splits['train'].num_examples
n_train     = int(total * 0.80)
n_val       = total - n_train

print(f"  Total samples  : {total:,}")
print(f"  Train split    : {n_train:,}  (80%)")
print(f"  Val split      : {n_val:,}  (20%)")
print(f"  Classes        : {CLASS_NAMES}")

# ── STEP 2: Preprocess & Build tf.data pipeline ──────────────
print(f"\n🔧 STEP 2: Preprocessing...")

def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0
    return image, label

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

train_ds = (full_ds
            .take(n_train)
            .map(preprocess,  num_parallel_calls=AUTOTUNE)
            .map(augment,     num_parallel_calls=AUTOTUNE)
            .shuffle(1000)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

val_ds = (full_ds
          .skip(n_train)
          .map(preprocess, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

print(f"  Normalized to 0–1 ✅")
print(f"  Resized to {IMG_SIZE}×{IMG_SIZE} ✅")
print(f"  Data augmentation: flip, brightness, contrast ✅")

# ── STEP 3: Visualize Samples ────────────────────────────────
print(f"\n🖼️  STEP 3: Saving Sample Images...")

sample_imgs, sample_labels = next(iter(train_ds.take(1)))

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Cats vs Dogs – Training Samples', fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_imgs[i].numpy())
    ax.set_title(CLASS_NAMES[int(sample_labels[i].numpy())],
                 fontsize=11, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
print("  ✅ Saved: sample_images.png")

# ── STEP 4: Build Transfer Learning Model ────────────────────
print(f"\n🏗️  STEP 4: Building Model (MobileNetV2 backbone)...")

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False   # Freeze all base layers

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=False)
x       = GlobalAveragePooling2D()(x)
x       = Dense(256, activation='relu')(x)
x       = Dropout(0.4)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs, name="CatsDogsTransfer")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

total_p     = model.count_params()
trainable_p = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f"  Base     : MobileNetV2 frozen (ImageNet weights)")
print(f"  Total    : {total_p:,} parameters")
print(f"  Trainable: {trainable_p:,} ({trainable_p/total_p*100:.1f}%) — head only")

# ── STEP 5: Train Phase 1 – Feature Extraction ───────────────
print(f"\n🚀 STEP 5: Training Phase 1 – Feature Extraction...")

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=2, verbose=1, min_lr=1e-7)
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)
e1 = len(history1.history['loss'])
best_phase1 = max(history1.history['val_accuracy'])
print(f"\n  ✅ Phase 1 done — {e1} epochs | Best val acc: {best_phase1*100:.2f}%")

# ── STEP 6: Train Phase 2 – Fine-Tuning ──────────────────────
print(f"\n🔧 STEP 6: Phase 2 – Fine-Tuning top 30 base layers...")

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)
e2 = len(history2.history['loss'])
best_phase2 = max(history2.history['val_accuracy'])
print(f"\n  ✅ Phase 2 done — {e2} epochs | Best val acc: {best_phase2*100:.2f}%")

# ── STEP 7: Evaluate ─────────────────────────────────────────
print(f"\n📊 STEP 7: Final Evaluation...")
test_loss, test_acc = model.evaluate(val_ds, verbose=0)

print(f"\n  Phase 1 Best Val Accuracy : {best_phase1*100:.2f}%")
print(f"  Phase 2 Best Val Accuracy : {best_phase2*100:.2f}%")
print(f"  Final   Val Accuracy      : {test_acc*100:.2f}%")
print(f"  Final   Val Loss          : {test_loss:.4f}")

# ── STEP 8: Training Curves ──────────────────────────────────
print(f"\n📈 STEP 8: Plotting Training Curves...")

acc      = history1.history['accuracy']     + history2.history['accuracy']
val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss     = history1.history['loss']         + history2.history['loss']
val_loss = history1.history['val_loss']     + history2.history['val_loss']
ep       = range(1, len(acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Transfer Learning – Training History  (Val Acc: {test_acc*100:.2f}%)',
             fontsize=13, fontweight='bold')

axes[0].plot(ep, acc,     'b-o', ms=4, label='Train Acc')
axes[0].plot(ep, val_acc, 'r-o', ms=4, label='Val Acc')
axes[0].axvline(x=e1 + 0.5, color='gray', linestyle='--',
                alpha=0.8, label=f'Fine-tune starts (ep {e1+1})')
axes[0].axhline(y=0.90, color='green', linestyle=':', alpha=0.7, label='90% target')
axes[0].set_title('Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, loss,     'b-o', ms=4, label='Train Loss')
axes[1].plot(ep, val_loss, 'r-o', ms=4, label='Val Loss')
axes[1].axvline(x=e1 + 0.5, color='gray', linestyle='--', alpha=0.8)
axes[1].set_title('Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
print("  ✅ Saved: training_curves.png")

# ── STEP 9: Sample Predictions ───────────────────────────────
print(f"\n🔍 STEP 9: Sample Predictions...")

pred_imgs, pred_true = next(iter(val_ds.take(1)))
pred_imgs  = pred_imgs[:15]
pred_true  = pred_true[:15].numpy()
probs      = model.predict(pred_imgs, verbose=0).flatten()
pred_class = (probs > 0.5).astype(int)

fig, axes = plt.subplots(3, 5, figsize=(16, 10))
fig.suptitle(f'Cats vs Dogs – Sample Predictions  (Val Acc: {test_acc*100:.2f}%)',
             fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(pred_imgs[i].numpy())
    actual  = CLASS_NAMES[int(pred_true[i])]
    pred    = CLASS_NAMES[pred_class[i]]
    conf    = probs[i] if pred_class[i] == 1 else 1 - probs[i]
    correct = pred_class[i] == int(pred_true[i])
    ax.set_title(f'{"✓" if correct else "✗"} {pred}\nTrue:{actual} ({conf*100:.0f}%)',
                 fontsize=8, color='green' if correct else 'red')
    ax.axis('off')
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
print("  ✅ Saved: sample_predictions.png")

print(f"\n  {'#':<4} {'Actual':<8} {'Predicted':<10} {'Conf':>7} {'Result'}")
print(f"  {'-'*38}")
for i in range(15):
    a    = CLASS_NAMES[int(pred_true[i])]
    p    = CLASS_NAMES[pred_class[i]]
    conf = probs[i] if pred_class[i] == 1 else 1 - probs[i]
    mark = "✅" if pred_class[i] == int(pred_true[i]) else "❌"
    print(f"  {i+1:<4} {a:<8} {p:<10} {conf*100:>6.1f}% {mark}")

print(f"\n{'=' * 64}")
print(f"  ✅  Transfer Learning Classifier Complete!")
print(f"  🎯  Final Validation Accuracy : {test_acc*100:.2f}%")
print(f"{'=' * 64}")


   🐱🐶  CATS vs DOGS – TRANSFER LEARNING CLASSIFIER  🐱🐶

📦 STEP 1: Loading Cats vs Dogs Dataset (TFDS)...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.GRN4CF_4.0.1/cats_vs_dogs-train.tfrecord-[0-9][0-9…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.
  Total samples  : 23,262
  Train split    : 18,609  (80%)
  Val split      : 4,653  (20%)
  Classes        : ['Cat', 'Dog']

🔧 STEP 2: Preprocessing...
  Normalized to 0–1 ✅
  Resized to 160×160 ✅
  Data augmentation: flip, brightness, contrast ✅

🖼️  STEP 3: Saving Sample Images...
  ✅ Saved: sample_images.png

🏗️  STEP 4: Building Model (MobileNetV2 backbone)...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  Base     : MobileNetV2 frozen (ImageNet weights)
  Total    : 2,586,177 parameters
  Trainable: 328,193 (12.7%) — head only

🚀 STEP 5: Training Phase 1 – Feature Extraction...
Epoch 1/20
582/582 ━━━━━━━━━━━━━━━━━━━━ 124s 161ms/step - accuracy: 0.9661 - loss: 0.0898 - val_accuracy: 0.9706 - val_loss: 0.0733 - learning_rate: 0.0010
Epoch 2/20
582/582 ━━━━━━━━━━━━━━━━━━━━ 77s 88ms/step - accuracy: 0.9774 - loss: 0.0605 - val_accuracy: 0.9800 - val